# Risk-controlling prediction sets for image regression

This notebook implements RCPS for the **expected fraction of missed pixels per image**. It trains a compact image reconstruction CNN, forms nested pixelwise intervals around each prediction, calibrates their expected image-level loss, and evaluates the result on untouched test images.

The statistical sample is a set of image pairs. Each image contributes one scalar loss in [0,1]; pixels inside an image can be dependent. This controls mean fractional pixel miss risk, not the chance that every pixel in one image is covered.

The calibration follows the monotone-family RCPS procedure from Bates et al. It uses a single calibration set for upper confidence bounds at each pre-fixed interval radius. It compares Hoeffding and WSR bounds. Set DELTA=0.10 below for a 90% risk certificate when the bound is below ALPHA.

The targets are LoDoPaB-derived images; FBP inputs are simulated from sparse-view noisy projections. This is an exploratory image-level experiment, not official-measurement or patient-level validation. Run cells in order. A GPU is optional.

In [ ]:
%pip -q install huggingface_hub scikit-image
import math
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from skimage.transform import radon, iradon, resize
from huggingface_hub import snapshot_download

SEED, SIZE = 21, 128
N_TRAIN, N_CAL, N_TEST = 700, 300, 100
N_ANGLES, NOISE_SNR_DB = 50, 30.0
ALPHAS = [0.01, 0.05, 0.10, 0.20]
EPOCHS, BATCH_SIZE = 5, 32
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
rng = np.random.default_rng(SEED); torch.manual_seed(SEED)
print('Device:', device)

In [ ]:
# Download only train/test data folders; skip the optional OOD images.
DATA_DIR = Path(snapshot_download(
    repo_id='AmirEhsan1995/lodopab-ct-glimpse', repo_type='dataset',
    allow_patterns=['train/**', 'test/**']))

def files_for(split):
    paths = sorted(p for p in DATA_DIR.rglob('*.npy') if split in p.parts)
    if not paths:
        raise FileNotFoundError(f'No .npy images for {split} under {DATA_DIR}; inspect mirror layout.')
    return paths

train_files, test_files = files_for('train'), files_for('test')
print('Images:', len(train_files), 'train,', len(test_files), 'test')
assert len(train_files) >= N_TRAIN + N_CAL and len(test_files) >= N_TEST
order = rng.permutation(len(train_files))
fit_files = [train_files[i] for i in order[:N_TRAIN]]
cal_files = [train_files[i] for i in order[N_TRAIN:N_TRAIN + N_CAL]]
test_order = rng.permutation(len(test_files))
heldout_files = [test_files[i] for i in test_order[:N_TEST]]

def load_targets(paths):
    out = []
    for p in paths:
        im = np.asarray(np.load(p), dtype=np.float32).squeeze()
        if im.ndim != 2: raise ValueError(f'Expected 2-D image, got {im.shape}: {p}')
        im = resize(im, (SIZE, SIZE), order=1, preserve_range=True, anti_aliasing=True)
        out.append(np.clip(im, 0, 1).astype(np.float32))
    return np.stack(out)

Y_fit, Y_cal, Y_test = map(load_targets, [fit_files, cal_files, heldout_files])
print('Targets:', Y_fit.shape, Y_cal.shape, Y_test.shape)

In [ ]:
# Simulate sparse-view measurements and reconstruct with filtered back projection.
angles = np.linspace(0, 180, N_ANGLES, endpoint=False)
def make_fbp_inputs(targets, seed):
    local_rng = np.random.default_rng(seed); result = []
    for i, im in enumerate(targets):
        sino = radon(im, theta=angles, circle=False).astype(np.float32)
        std = (10 ** (-NOISE_SNR_DB / 20)) * np.linalg.norm(sino) / np.sqrt(sino.size)
        noisy = sino + local_rng.normal(0, std, sino.shape).astype(np.float32)
        result.append(iradon(noisy, theta=angles, circle=False, output_size=SIZE).astype(np.float32))
        if (i + 1) % 100 == 0: print(f'FBP {i + 1}/{len(targets)}')
    return np.stack(result)
X_fit = make_fbp_inputs(Y_fit, SEED + 1)
X_cal = make_fbp_inputs(Y_cal, SEED + 2)
X_test = make_fbp_inputs(Y_test, SEED + 3)
fig, ax = plt.subplots(1, 2, figsize=(7, 3))
ax[0].imshow(X_fit[0], cmap='gray'); ax[0].set_title('Noisy sparse-view FBP')
ax[1].imshow(Y_fit[0], cmap='gray', vmin=0, vmax=1); ax[1].set_title('LoDoPaB-derived target')
for a in ax: a.axis('off')
plt.tight_layout()

In [ ]:
class SmallCNN(nn.Module):
    def __init__(self, width=32):
        super().__init__()
        self.net = nn.Sequential(nn.Conv2d(1,width,3,padding=1), nn.ReLU(),
            nn.Conv2d(width,width,3,padding=1), nn.ReLU(),
            nn.Conv2d(width,width,3,padding=1), nn.ReLU(),
            nn.Conv2d(width,1,3,padding=1))
    def forward(self, x): return x + self.net(x)

def tensor4(a): return torch.from_numpy(a).float().unsqueeze(1)
loader = DataLoader(TensorDataset(tensor4(X_fit), tensor4(Y_fit)),
                    batch_size=BATCH_SIZE, shuffle=True)
model = SmallCNN().to(device)
opt = torch.optim.Adam(model.parameters(), lr=1e-3)
for epoch in range(EPOCHS):
    model.train(); total = 0.0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        loss = F.mse_loss(model(x), y)
        opt.zero_grad(set_to_none=True); loss.backward(); opt.step()
        total += loss.item() * len(x)
    print(f'Epoch {epoch+1}/{EPOCHS}: train MSE={total/len(loader.dataset):.6f}')

@torch.no_grad()
def predict(a):
    model.eval(); out=[]; x=tensor4(a)
    for i in range(0, len(x), BATCH_SIZE): out.append(model(x[i:i+BATCH_SIZE].to(device)).cpu())
    return torch.cat(out)
pred_cal, pred_test = predict(X_cal), predict(X_test)
# Clip centers to the known target range so the residual score exactly
# matches the intervals constructed below.
pred_cal = pred_cal.clamp(0, 1)
pred_test = pred_test.clamp(0, 1)

In [ ]:
# RCPS bounds and monotone-family calibration.
# These definitions keep the notebook runnable on its own; matching reusable
# functions live in src/rcps/ in this repository.
def hoeffding_ucb(losses, delta):
    losses = np.asarray(losses, dtype=float)
    return float(min(1.0, losses.mean() + math.sqrt(math.log(1/delta)/(2*len(losses)))))

def wsr_log_capital(losses, risk, delta):
    n = len(losses); running_sum = 0.0; squared_deviations = 0.0
    previous_variance = 0.25; log_capital = 0.0
    for i, loss in enumerate(losses, start=1):
        running_sum += float(loss)
        mean_i = (0.5 + running_sum) / (i + 1)
        squared_deviations += (float(loss) - mean_i) ** 2
        variance_i = (0.25 + squared_deviations) / (i + 1)
        nu = min(1.0, math.sqrt(2*math.log(1/delta)/(n*previous_variance)))
        factor = 1 + nu*(risk-float(loss))
        if factor <= 0: return -math.inf
        log_capital += math.log(factor); previous_variance = variance_i
    return log_capital

def wsr_ucb(losses, delta, iterations=64):
    losses = np.asarray(losses, dtype=float); lo, hi = 0.0, 1.0
    boundary = math.log(1/delta)
    for _ in range(iterations):
        mid = (lo+hi)/2
        if wsr_log_capital(losses, mid, delta) > boundary: hi = mid
        else: lo = mid
    return hi

def calibrate(loss_table, radii, alpha, delta, method):
    bound_fn = {'hoeffding':hoeffding_ucb, 'wsr':wsr_ucb}[method]
    empirical = loss_table.mean(axis=0)
    ucbs = np.array([bound_fn(loss_table[:,j], delta) for j in range(len(radii))])
    suffix_max = np.maximum.accumulate(ucbs[::-1])[::-1]
    passing = np.flatnonzero(suffix_max < alpha)
    return empirical, ucbs, (int(passing[0]) if len(passing) else None)

# Per-image fractional pixel-miss loss for each pre-fixed interval radius.
# Every row is one calibration image; every column is one nested interval.
RISK_LEVELS = [0.01, 0.05, 0.10, 0.20]
DELTA = 0.10
RADIUS_GRID = np.linspace(0.0, 1.0, 201)

pred_cal_np = pred_cal.numpy()
cal_losses = np.column_stack([
    (((Y_cal[:, None, :, :] < np.clip(pred_cal_np-r, 0, 1)) |
      (Y_cal[:, None, :, :] > np.clip(pred_cal_np+r, 0, 1)))
     .reshape(len(Y_cal), -1).mean(axis=1)) for r in RADIUS_GRID
])

rcps_results = {}
print('method | alpha risk | selected radius | calibration mean loss | risk UCB | test pixel coverage | test whole-image coverage | mean width | status')
for method in ['hoeffding', 'wsr']:
    for alpha_risk in RISK_LEVELS:
        empirical, ucbs, selected = calibrate(cal_losses, RADIUS_GRID, alpha_risk, DELTA, method)
        if selected is None:
            print(f'{method} | {alpha_risk:.2f} | -- | -- | -- | -- | -- | -- | no certificate')
            continue
        lam = float(RADIUS_GRID[selected])
        k = selected
        lo = torch.clamp(pred_test - lam, 0, 1)
        hi = torch.clamp(pred_test + lam, 0, 1)
        yt = tensor4(Y_test)
        inside = (yt >= lo) & (yt <= hi)
        pixel_cov = inside.float().mean().item()
        image_cov = inside.flatten(1).all(1).float().mean().item()
        width = (hi-lo).mean().item()
        rcps_results[(method, alpha_risk)] = {
            'radius': lam, 'lower': lo, 'upper': hi,
            'calibration_risk': empirical[k],
            'ucb': ucbs[k], 'pixel_coverage': pixel_cov,
            'image_coverage': image_cov, 'mean_width': width,
        }
        print(f'{method} | {alpha_risk:.2f} | {lam:.3f} | {empirical[k]:.4f} | {ucbs[k]:.4f} | {pixel_cov:.4f} | {image_cov:.3f} | {width:.4f} | certified')

# Plot the calibration empirical risks and UCBs for a representative target.
alpha_plot = 0.10
fig, ax = plt.subplots(figsize=(7, 4))
for method in ['hoeffding', 'wsr']:
    empirical, ucbs, _ = calibrate(cal_losses, RADIUS_GRID, alpha_plot, DELTA, method)
    ax.plot(RADIUS_GRID, ucbs, label=f'{method} risk UCB')
    ax.plot(RADIUS_GRID, empirical, '--', alpha=0.7, label=f'{method} empirical risk')
ax.axhline(alpha_plot, color='black', linestyle=':', label='alpha = 0.10')
ax.set(xlabel='Interval radius (larger means wider sets)', ylabel='Expected fraction of missed pixels', title='RCPS calibration on the calibration split')
ax.set_ylim(0, 1); ax.grid(alpha=0.25); ax.legend(); plt.show()

## What the output means

- **alpha** is the allowed expected fraction of missed pixels on a fresh image. It is not a per-image or pixelwise confidence level.
- **delta** is the probability the risk upper bound fails over repeated calibration samples. With delta = 0.10, a bound below alpha gives a 90% confidence certificate under the sampling assumptions.
- **Calibration mean loss** is observed average fractional pixel miss across calibration images. **Risk UCB** is the high-probability upper bound on population mean loss. A certificate is reported only if the monotone-family rule accepts a radius.
- **Test pixel coverage** is the empirical fraction of test pixels inside intervals; its complement equals the mean fractional-pixel miss loss because images are the same size. **Test whole-image coverage** is a different diagnostic and is not the controlled risk.
- **Mean width** describes interval size, not statistical validity. Test metrics are diagnostics and never participate in calibration.

For every pre-fixed radius, the code computes a one-sided bound on the mean of image losses. It then selects the smallest radius for which the bound at that radius and every larger radius is below alpha. The larger sets are nested and their losses are nonincreasing. This monotone-family selection is the paper's RCPS approach; it avoids an additional calibration split or a union bound over all radii. Each alpha row is an individual 1-delta claim; the four alpha rows are not jointly certified.

### Assumptions and limits

The trained model, loss, preprocessing, and radius grid are fixed before inspecting calibration labels. The concentration bounds here assume calibration image pairs are IID with future image pairs, conditional on the fitted model; exchangeability alone does not justify these mean-risk bounds. Pixel dependence within an image is allowed because pixels are averaged into one loss. Correlated images, such as multiple slices from one patient, can violate image-level independence; this small mirror does not document patient grouping. Inputs here are simulated sparse-view FBP, not official LoDoPaB measurement data.

### Why include the Gaussian chi-square baseline?

It is deliberately not reported as a competing RCPS score: a chi-square/Mahalanobis ellipsoid relies on a multivariate Gaussian residual model and controls a different geometric event. Applying it to this bounded fractional-pixel-risk target would not be an apples-to-apples baseline. See [the paper](https://arxiv.org/abs/2101.02703) and [method notes](../docs/method.md).

In [ ]:
# Visualize an untouched test example for one certified configuration.
key = ('wsr', 0.10)
if key not in rcps_results:
    print('No certified WSR radius for alpha=0.10 at this calibration size/delta.')
else:
    rr = rcps_results[key]
    j = 0
    target = Y_test[j]
    center = pred_test[j, 0].numpy()
    lower = rr['lower'][j, 0].numpy()
    upper = rr['upper'][j, 0].numpy()
    missed = (target < lower) | (target > upper)
    fig, ax = plt.subplots(1, 5, figsize=(15, 3))
    ax[0].imshow(X_test[j], cmap='gray'); ax[0].set_title('Noisy sparse-view FBP')
    ax[1].imshow(target, cmap='gray', vmin=0, vmax=1); ax[1].set_title('Target')
    ax[2].imshow(center, cmap='gray', vmin=0, vmax=1); ax[2].set_title('Prediction')
    im = ax[3].imshow(upper-lower, cmap='magma'); ax[3].set_title('Interval width')
    fig.colorbar(im, ax=ax[3], fraction=0.046)
    ax[4].imshow(missed, cmap='Reds'); ax[4].set_title('Missed pixels')
    for a in ax: a.axis('off')
    plt.suptitle(f'WSR RCPS alpha=0.10, radius={rr["radius"]:.3f}, UCB={rr["ucb"]:.3f}')
    plt.tight_layout(); plt.show()